## 02 embedding 2.0 concat 

In [1]:
import torch
import h5py
from tqdm import tqdm
import  esm

In [2]:
import os
import sys
from pathlib import Path

def find_project_root(start: Path, marker: str = "src") -> Path:
    """
    Walk upward from the notebook's directory until we find
    a folder containing 'src'. That folder is the project root.
    """
    current = start.resolve()
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent
    raise RuntimeError("Project root not found. Make sure 'src/' exists.")

# Detect project root automatically
PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Detected project root:", PROJECT_ROOT)


Detected project root: /home/yi-jin/Documents/CATA6-protein-prediction-4staroverook


In [3]:
%cd "$PROJECT_ROOT"
!ls

/home/yi-jin/Documents/CATA6-protein-prediction-4staroverook
configs  data  LICENSE	logs  notebooks  README.md  src  tmpDir


/home/yi-jin/anaconda3/envs/cafa6/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
import os

data_raw_path = PROJECT_ROOT / "data" / "raw"
print("data/raw exists:", os.path.exists(data_raw_path))
print(os.listdir(data_raw_path) if os.path.exists(data_raw_path) else "MISSING")

train_fasta_path = PROJECT_ROOT / "data" / "raw" / "Train"
print(os.path.exists(train_fasta_path))
print(os.listdir(train_fasta_path) if os.path.exists(train_fasta_path) else "MISSING")


data/raw exists: True
['sample_submission.tsv', 'Test', 'Train', '.gitkeep', 'IA.tsv', '.DS_Store']
True
['go-basic.obo', 'train_terms.tsv', '.ipynb_checkpoints', 'train_taxonomy.tsv', 'train_sequences.fasta']


### Step1 load fasta, ID splits 

In [4]:

from src.preprocessing.prepare_embedding import (
    load_train_val_test_sequences,
    load_length_bins
)

train_seqs, val_seqs, test_seqs = load_train_val_test_sequences(PROJECT_ROOT)
len(train_seqs), len(val_seqs), len(test_seqs)




(70093, 12311, 224309)

In [5]:
# confirm binning 
train_bins, val_bins, test_bins = load_length_bins(train_seqs, val_seqs, test_seqs)

for name, bin_dict in [("Train", train_bins), ("Val", val_bins), ("Test", test_bins)]:
    print(f"=== {name} ===")
    for k, v in bin_dict.items():
        print(k, len(v))


=== Train ===
short_<=1022 63557
mid_1023_2048 5455
long_2049_5000 1017
ultra_>5000 64
=== Val ===
short_<=1022 11192
mid_1023_2048 927
long_2049_5000 173
ultra_>5000 19
=== Test ===
short_<=1022 210921
mid_1023_2048 11172
long_2049_5000 2066
ultra_>5000 150


### Step 2 Load ESM2 Model 

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
model = model.to(device)
model.eval()

batch_converter = alphabet.get_batch_converter()


Device: cuda


In [7]:
#### run a minitest to confirm the setting
train_seqs, val_seqs, test_seqs = load_train_val_test_sequences(PROJECT_ROOT)

# mini test: first 10 sequences
mini = train_seqs[:10]
len(mini), mini[:2]


(10,
 [('Q9I6J2',
   'MNSQITNAKTREWQALSRDHHLPPFTDYKQLNEKGARIITKAEGVYIWDSEGNKILDAMAGLWCVNVGYGREELVQAATRQMRELPFYNLFFQTAHPPVVELAKAIADVAPEGMNHVFFTGSGSEANDTVLRMVRHYWATKGQPQKKVVIGRWNGYHGSTVAGVSLGGMKALHEQGDFPIPGIVHIAQPYWYGEGGDMSPDEFGVWAAEQLEKKILEVGEENVAAFIAEPIQGAGGVIVPPDTYWPKIREILAKYDILFIADEVICGFGRTGEWFGSQYYGNAPDLMPIAKGLTSGYIPMGGVVVRDEIVEVLNQGGEFYHGFTYSGHPVAAAVALENIRILREEKIIEKVKAETAPYLQKRWQELADHPLVGEARGVGMVAALELVKNKKTRERFTDKGVGMLCREHCFRNGLIMRAVGDTMIISPPLVIDPSQIDELITLARKCLDQTAAAVLA'),
  ('Q9D311',
   'MTAWDGVLPFYPQPRHAASFSVPLLIVILVFLSLAASFLFILPGIRGHSRWFWLVRVLLSLFIGAEIVAVHFSGDWFVGRVWTNTSYKAFSPSRVQVHVGLHVGLAGVNITLRGTPRQQLNETIDYNERFTWRLNEDYTKEYVHALEKGLPDPVLYLAEKFTPSSPCGLYHQYHLAGHYAAATLWVAFCFWIIANALLSMPAPLYGGLALLTTGAFTLFGVFAFASISSVPLCHFRLGSAVLTPYYGASFWLTLATGILSLLLGGAVVILHYTRPSALRSFLDLSVKDCSNQAKGNSPLTLNNPQHEQLKSPDLNITTLL')])

In [8]:
from src.preprocessing.esm_concat import embed_concat_batch, embed_bin_concat
test_batch = mini[:4]
emb_dict = embed_concat_batch(model, batch_converter, device, test_batch)

list(emb_dict.keys()), emb_dict[next(iter(emb_dict))].shape


(['Q9I6J2', 'Q9D311', 'Q6ENH4', 'Q05594'], (2560,))

### Step 3 Embedding 

In [10]:
from src.preprocessing.esm_concat import run_concat_embedding
from pathlib import Path

# Detect project root automatically

BASE = PROJECT_ROOT

# Choose a clear output file name
OUTPUT = f"{BASE}/data/embeddings/esm2_650M_trainval_concat_2560.h5"

print("BASE:", BASE)
print("OUTPUT:", OUTPUT)

run_concat_embedding(BASE, OUTPUT, mode= "trainval")


BASE: /home/yi-jin/Documents/CATA6-protein-prediction-4staroverook
OUTPUT: /home/yi-jin/Documents/CATA6-protein-prediction-4staroverook/data/embeddings/esm2_650M_trainval_concat_2560.h5
Running on device: cuda
Loading ESM2-650M model...
Loading sequences...
Train: 70093  Val: 12311  Test: 224309

 STARTING TRAIN EMBEDDING 

Processing bin: short_<=1022
  Remaining in this bin: 0


0it [00:00, ?it/s]


Processing bin: mid_1023_2048
  Remaining in this bin: 0


0it [00:00, ?it/s]


Processing bin: long_2049_5000
  Remaining in this bin: 0


...

Validation bin: long_2049_5000
  Remaining in this bin: 0


0it [00:00, ?it/s]


Validation bin: ultra_>5000
  Remaining in this bin: 0


0it [00:00, ?it/s]



=== CONCAT EMBEDDING COMPLETE ===
Saved to: /home/yi-jin/Documents/CATA6-protein-prediction-4staroverook/data/embeddings/esm2_650M_trainval_concat_2560.h5


In [ ]:
# after completion, check out put. 

In [ ]:
# run test seperately 

In [9]:
from src.preprocessing.esm_concat import run_concat_embedding
from pathlib import Path

OUTPUT_TEST = f"{PROJECT_ROOT}/data/embeddings/esm2_650M_test_concat_2560.h5"
run_concat_embedding(PROJECT_ROOT, OUTPUT_TEST, mode="test")


Running on device: cuda
Loading ESM2-650M model...
Loading sequences...
Train: 70093 | Val: 12311 | Test: 224309

 STARTING TEST EMBEDDING 

[Test] Bin: short_<=1022
  Remaining in this bin: 210921


100%|███████████████████████████████████████████████████████████████████████| 26366/26366 [2:30:35<00:00,  2.92it/s]


[Test] Bin: mid_1023_2048
  Remaining in this bin: 11172


100%|███████████████████████████████████████████████████████████████████████████| 1397/1397 [11:44<00:00,  1.98it/s]


[Test] Bin: long_2049_5000
  Remaining in this bin: 2066


100%|█████████████████████████████████████████████████████████████████████████████| 259/259 [02:11<00:00,  1.97it/s]


[Test] Bin: ultra_>5000
  Remaining in this bin: 150


100%|███████████████████████████████████████████████████████████████████████████████| 19/19 [00:09<00:00,  1.99it/s]


=== CONCAT EMBEDDING COMPLETE ===
Saved to: /home/yi-jin/Documents/CATA6-protein-prediction-4staroverook/data/embeddings/esm2_650M_test_concat_2560.h5
